<table border=0 width="100%"><tr><td><p align="left"><font size=5><B>数据分析与可视化</B></font></p></td><td><font size=3><B>demo-streamlit(郑海超)</B></font></td></tr></table>

# 内容
- 数据分析应用展示工具：streamlit

# 数据分析应用展示工具：streamlit

- 这个工具除了在我们课程使用，其他课程项目展示也有用

https://docs.streamlit.io/get-started

!pip install streamlit

案例展示

streamlit hello

运行streamlit应用代码py文件

streamlit run your_script.py

## Basic concepts of Streamlit
https://docs.streamlit.io/get-started/fundamentals/main-concepts

```python
"""
# My first app
Here's our first attempt at using data to create a table:
"""

import streamlit as st
import pandas as pd
df = pd.DataFrame({
  'first column': [1, 2, 3, 4],
  'second column': [10, 20, 30, 40]
})
```

df

````python
import streamlit as st
import numpy as np
import pandas as pd

dataframe = pd.DataFrame(
    np.random.randn(10, 5),
    columns=('col %d' % i for i in range(5)))
st.table(dataframe) # 不运行修改
```

```python
import streamlit as st
import numpy as np
import pandas as pd

chart_data = pd.DataFrame(
     np.random.randn(20, 3),
     columns=['a', 'b', 'c'])

st.line_chart(chart_data)
```

```python
import streamlit as st
x = st.slider('x')  # 👈 this is a widget
st.write(x, 'squared is', x * x)
````

```python
import streamlit as st
st.text_input("Your name", key="name")

# You can access the value at any point with:
st.session_state.name
```         

```python
import streamlit as st
import pandas as pd

df = pd.DataFrame({
    'first column': [1, 2, 3, 4],
    'second column': [10, 20, 30, 40]
    })

option = st.selectbox(
    'Which number do you like best?',
     df['first column'])

'You selected: ', option
```

```python
import time
import streamlit as st
import time

'Starting a long computation...'

# Add a placeholder
latest_iteration = st.empty()
bar = st.progress(0)

for i in range(100):
  # Update the progress bar with each iteration.
  latest_iteration.text(f'Iteration {i+1}')
  bar.progress(i + 1)
  time.sleep(0.1)

'...and now we\'re done!'
```

## 案例：股票价格预测

### 获取股票的每日指标

In [ ]:
import tushare as ts

https://tushare.pro/document/2?doc_id=32

In [ ]:
import tushare as ts
# 为了批改方便，我这里将tushare token直接放到代码中
token = ""
ts.set_token(token=token)
pro = ts.pro_api()

In [ ]:
pro = ts.pro_api()
# 下面trade_date是需要替换为最近一天的日期字符串
df = pro.daily_basic(ts_code='000001.SZ', trade_date='20180726', fields='ts_code,trade_date,turnover_rate,volume_ratio,pe,pb')


In [ ]:
df

### 获取多天的指标数据

In [ ]:
# 初始化空的DataFrame用于存储结果
df = pd.DataFrame()

# 从今天倒推5天，获取日期列表
today = pd.Timestamp.now()
for i in range(5):
      # 计算日期
      target_date = today - pd.Timedelta(days=i)
      # 格式化为YYYYMMDD格式
      trade_date = target_date.strftime('%Y%m%d')
      print(trade_date)

      try:
            # 获取指定日期的数据
            daily_data = pro.daily_basic(ts_code="000001.SZ", trade_date=trade_date,
                                    fields='ts_code,trade_date,turnover_rate,volume_ratio,pe,pb')
            # 避免空数据或全NA列的concat警告
            if not daily_data.empty and not all(daily_data.isna().all()):
                  df = pd.concat([df, daily_data], ignore_index=True)
      except Exception as e:
      # 如果某一天没有数据，继续获取下一天
            continue


In [ ]:
df

### chat with AI

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)  # 强制重新加载环境变量，确保获取最新值
import os
# 课堂默认用 DeepSeek；换其他兼容 OpenAI 接口的厂商时，换掉环境变量名即可
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")


In [ ]:
def llm_chat(prompt):
    from openai import OpenAI

    # DeepSeek 兼容 OpenAI 接口：换厂商只需改 base_url、模型名和 API Key 环境变量
    # 例如豆包：base_url="https://ark.cn-beijing.volces.com/api/v3"，模型名换豆包的接入点
    client = OpenAI(
        base_url="https://api.deepseek.com",   # DeepSeek API 地址
        api_key=DEEPSEEK_API_KEY,              # 从环境变量中获取您的 API Key
    )

    # Non-streaming:
    completion = client.chat.completions.create(
        model="deepseek-chat",                 # DeepSeek 对话模型
        messages=[
            {"role": "system", "content": "你是人工智能助手"},
            {"role": "user", "content": prompt},
        ],
    )
    return completion.choices[0].message.content


In [ ]:
answer = llm_chat("hello")

In [ ]:
print(answer)

### 构造预测prompt，并预测

In [ ]:
firm_name = "平安银行"

In [ ]:
resolution_criteria = "预测的判断标准：股票价格在随后的第五个交易日是否高于或低于当前价格。"
task_description = """未来该股票的价格在随后的第五个交易日会上涨还是下跌？"""
question = f"will future price of this stock {firm_name} moves up or down in the subsequent fifth trading day?"


In [ ]:
materials = df.to_dict(orient='records')
materials = str(materials)
materials

In [ ]:
prompt = f"""
    You are an expert in future event prediction.
        - Question:{question}
        - Resolution Criteria:{resolution_criteria}

    We have retrieved the following Materials for this question (Special Note: Some indicators in the materials may be percentages (%) and not decimals):
        - Materials: {materials}
    
    Solve the problem(question) step by step as required by the intructions.
     
    Instructions:
       1. Given the above question, rephrase and expand it to help you do better answering. Maintain all information in the original question.
    {{ Insert rephrased and expanded question. }}
    2. Based on the materials provided, extract and provide a few reasons why the answer might be in each option (i.e., stock price moves up or moves down). Rate the strength of each reason for each option.
    {{ Insert your thoughts }}
    3. Based on the materials provided, extract and provide a few reasons why the answer might be not in each option (i.e., stock price moves up or moves down). Rate the strength of each reason for each option.
    {{ Insert your thoughts }}
    4. Aggregate your considerations. Think like a superforecaster (e.g. Nate Silver).
    {{ Insert your aggregated considerations }}
    5. Output an initial probability (prediction) given steps 1-4.
    {{ Insert initial probability,  do not ouput initial probability in JSON format}}
    6. Reflect, and evaluate whether your calculated probability is excessively confident or not confident enough. Also, consider anything else that might affect the forecast that you did not before consider (e.g. base rate of the event).
    {{ Insert your thoughts }}
    7. Output your final probability prediction (the sum of up and down probabilities must equal to 1).
    {{ Insert your answer, your final prediction must be in JSON format with keys: up and down }}
      8. 回答问题用中文。
"""

In [ ]:
print(prompt)

In [ ]:
answer = llm_chat(prompt)
print(answer)

### 从返回的response中提取json数据

In [ ]:
def extract_json(response):
    """
    使用正则表达式，从返回的response中提取json数据
    """
    import re
    import json

    match = re.search(r'\{.*\}', response, re.DOTALL)
    if not match:
        raise ValueError("未在响应中找到 JSON 数据")
    json_str = match.group()
    return json.loads(json_str)



In [ ]:
answer = """
1. 重新表述并扩展后的问题：已知平安银行股票在2018年7月26日的相关指标数据，包括换手率为0.4854、量比为0.69、市盈率为6.8492、市净率为0.7898，要预测该股票在随后的第五个交易日的价格相较于当前价格是上涨还是下跌，判断标准为股票价格在随后的第五个交易日是否高于或低于当前价格。
2. 答案可能为股票价格上涨的原因及强度：
    - 市盈率6.8492相对较低，可能意味着股票被低估，有上涨潜力。强度：中等。因为较低的市盈率只是一个参考因素，市场情况复杂，不能仅凭此确定上涨。
    - 市净率0.7898较低，说明股票价格相对其净资产有一定优势，可能吸引投资者买入推动股价上升。强度：中等。同样，市净率低不必然导致股价上涨。
3. 答案可能不为股票价格上涨的原因及强度：
    - 换手率0.4854较低，表明市场交易不活跃，缺乏足够的资金推动股价上涨。强度：较强。低换手率往往意味着市场参与度不高，股价上涨动力不足。
    - 量比0.69小于1，显示当前成交量低于过去平均水平，上涨动能不足。强度：较强。量比低说明买盘不积极，不利于股价上升。
4. 综合考虑：虽然市盈率和市净率显示股票可能有一定的投资价值，但换手率和量比反映出市场交易不活跃和上涨动能不足。综合来看，股价下跌的可能性相对较大。
5. 初始概率：上涨概率30%，下跌概率70%。
6. 反思：初始概率的判断有一定依据，但可能存在一定的过度自信。因为仅依据这几个指标来判断股价走势是不够全面的，股票价格还受到宏观经济环境、行业发展趋势、公司基本面变化等多种因素的影响。而且没有考虑该股票价格上涨或下跌的历史概率（即基率）。
7. 最终概率预测：
{
    "up": 0.3,
    "down": 0.7
}
"""

In [ ]:
prediction = extract_json(answer)
prediction


### 使用streamlit构建应用展示 （尽量在课堂完成）

- 结合trae辅助编程实现
- API_KEY建议放在.env文件中
- 除了课堂上使用的daily_basic指标，你也可以从tushare获取其他有价值的数据，提高预测准确性

**下周要分享的同学，不需要将.env发给我QQ**

# 本周课后学习安排
- 完成实验 - 应用streamlit和大模型做一个预测股价波动方向的APP
- 文献阅读 - Are Monthly Seasonals Real? A Three Century Perspective